# Baseline Hybrid RAG (Sparse + Dense + Optional Rerank)

Clean, self-contained notebook implementing a working **hybrid RAG** pipeline:

1. Document loading & cleaning  
2. Chunking (tables kept intact)  
3. **Sparse** retriever (BM25)  
4. **Dense** retriever (MiniLM + Chroma)  
5. **Hybrid** = Ensemble(BM25 + Dense) capped at top-k  
6. **Hybrid + Rerank** = Hybrid → Cross-Encoder  
7. Generation with local Ollama (`qwen2.5:3b`)  
8. Evaluation on 50 history questions (token-F1, semantic sim, optional LLM-as-judge)

**Requirements**: Ollama with `qwen2.5:3b` pulled. GPU recommended but not required.


## 0. Environment & Installs


In [1]:
# GPU check (optional)
!nvidia-smi || true

Sat Oct 24 09:56:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Core packages – versions that work together with current LangChain 0.3.x
%pip install -q --upgrade pip
%pip install -q \
    "langchain>=0.3.0,<0.4" \
    "langchain-community>=0.3.0,<0.4" \
    "langchain-core>=0.3.0" \
    "langchain-text-splitters" \
    "langchain-ollama" \
    "langchain-huggingface" \
    rank-bm25 sentence-transformers chromadb \
    pypdf docx2txt \
    pandas numpy scikit-learn tqdm \
    rouge-score nltk rapidfuzz

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
langgraph-sdk 0.4.8 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.16 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.


## 1. Start Ollama (local LLM)


In [6]:
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package pci.ids.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselec

In [7]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [8]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log","w"), stderr=subprocess.STDOUT)
time.sleep(6)
!curl -s http://127.0.0.1:11434/api/version
!ollama pull qwen2.5:3b

{"version":"0.40.10"}


In [9]:
# 1. Is the CUDA library visible to Ollama?
!ollama ps

NAME    ID    SIZE    PROCESSOR    CONTEXT    RUNNER    UNTIL 


In [10]:
!tail -50 /tmp/ollama.log

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIJQ+t31+FEtBKlz3iAlM2qdAHLXuSHqqfUv8UUvWuJGC

time=2026-10-24T10:02:59.004Z level=INFO source=routes.go:2451 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_

In [11]:
!ollama list

NAME          ID              SIZE      MODIFIED           
qwen2.5:3b    357c53fb659c    1.9 GB    About a minute ago    


In [14]:
# Quick generation — this loads the model onto GPU
!curl -s http://127.0.0.1:11434/api/generate -d '{"model": "qwen2.5:3b", "prompt": "Say hello in one short sentence.", "stream": false}' | head -c 500

{"model":"qwen2.5:3b","created_at":"2026-10-24T10:08:25.245164895Z","response":"Hello! How can I assist you today?","done":true,"done_reason":"stop","context":[151644,8948,198,2610,525,1207,16948,11,3465,553,54364,14817,13,1446,525,264,10950,17847,13,151645,198,151644,872,198,45764,23811,304,825,2805,11652,13,151645,198,151644,77091,198,9707,0,2585,646,358,7789,498,3351,30],"total_duration":112268538516,"load_duration":64370589029,"prompt_eval_count":36,"prompt_eval_cached_count":0,"prompt_eval_

In [15]:
!ollama ps
!nvidia-smi

NAME          ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:3b    357c53fb659c    2.2 GB    100% GPU     4096       llamacpp    4 minutes from now    
Sat Oct 24 10:08:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             33W /  

## 2. Load & Clean Documents

Place your corpus under `DATA_DIR` (default `/content/TEST_RAG`).  
Supported extensions: `.pdf`, `.docx`, `.txt`, `.html`.

## 3. Chunking

Keep table-like blocks whole; split ordinary text with recursive character splitter.

ALL DONE VIA UNSTRUCTURED FREE API

In [20]:
%pip install -q "langchain-unstructured" "unstructured-client>=0.25"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.30 requires langchain-core<1.0.0,>=0.3.85, but you have langchain-core 1.7.0 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.7.0 which is incompatible.


In [21]:

from langchain_core.documents import Document


import os
from pathlib import Path
from langchain_unstructured import UnstructuredLoader
from langchain_core.documents import Document

In [ ]:


os.environ["UNSTRUCTURED_API_KEY"] = "API"   # <-- paste key

DATA_DIR = Path(os.environ.get("RAG_DATA_DIR", "/content/TEST_RAG"))
SUPPORTED = {".pdf", ".docx", ".doc", ".pptx", ".html", ".txt", ".md", ".eml"}

chunks: list[Document] = []

for path in sorted(DATA_DIR.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in SUPPORTED:
        continue
    print(f"Loading {path.name} ...")
    try:
        loader = UnstructuredLoader(
            file_path=str(path),
            partition_via_api=True,
            api_key=os.environ["UNSTRUCTURED_API_KEY"],
            strategy="hi_res",              # best quality (tables + layout)
            # strategy="fast",              # use this if free tier is slow / rate-limited
            chunking_strategy="by_title",   # groups under headings
            max_characters=2000,
            new_after_n_chars=1800,         # soft start of next chunk
            overlap=200,
            overlap_all=True,
            # languages=["eng"],            # optional
        )
        docs = loader.load()
        for d in docs:
            d.metadata["source"] = str(path)   # keep original path
        chunks.extend(docs)
        print(f"  → {len(docs)} chunks")
    except Exception as e:
        print(f"  ⚠️  skipped {path.name}: {type(e).__name__}: {e}")

# stable ids for later audits / eval
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(f"\n✅ Total chunks: {len(chunks)}")
if chunks:
    print("Sample metadata:", chunks[0].metadata)
    print("Sample text:", chunks[0].page_content[:300], "...")

Loading TEST_HISTORY_RAG_PDF.pdf ...
  → 46 chunks
Loading Table-Examples.pdf ...
  → 24 chunks

✅ Total chunks: 70
Sample metadata: {'source': '/content/TEST_RAG/TEST_HISTORY_RAG_PDF.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'orig_elements': 'eJy1V8uO2zYU/RXCa0vQ+9FNkb6ALloUSYAukmJAkVcWG0l0SWpct+i/95CSJm4yKNoBvPGYd3gv7+OcQ/rdnwcaaaLZPSh5+IIduqLo+obSqBVVGxVl10dt3fdRksuszpo8aZr6cGSHiRyX3HH4/HkQWhupZu7IhvXIr3pxDwOp0+BgyZKqgs9mvijpBljTomxhPWs1O+/37l2aNHF9ZFmexPkvR7avizSPk7BOizxunzGsHrAc7NU6mnwlP6nfaXxz5oIOf+EfkhwJp/T8IEZu7cPZ6A7bkrhN6rbBhl6N5K5n8r5f/fCTL1LZB/rdGS4cheY4s9AhFDKfFn4K1b470Hw6+KPPsDzMy9SR8dX5Qx3cveMbLRQflZ2Ymtm3i9FnYnyWzA3EXi/WKj6z1/Sox8Vn6I/YM3mr3BgK+HRQdZ6WXVVlUdnmTVTUVR01sioi2XVJz6lJury836DSJMbGsirjbJ1DWFd5sQ6uadq4+Xy97n/ZmJq2bvM7jymN2VtM5NWJmO7ZOjT29YAw9J9mIuusaJu8iDhv06hIZBe1rQCDRFdXuaC2rOQdyVPH5ZHVSR7XK3nCOs1hKIMhz9vnLavPC+lT52Vx57l8PwemnA09Kr1YJgZ+dmQYOsIMccl4h96se/SFTL+MTEni1o+xN0RST4Fv9NsCGrortnLHhDJiGdF4eUvL3kf2ob4zNIvhhpcrOj4zM7jNi

In [ ]:
chunks

## 4. Retrievers – Sparse, Dense, Hybrid, Hybrid+Rerank


In [25]:
# Align packages (run once)
%pip install -q -U \
  "langchain>=0.3.0,<0.4" \
  "langchain-community>=0.3.0,<0.4" \
  "langchain-core>=0.3.0" \
  "langchain-text-splitters" \
  sentence-transformers rank-bm25 chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-unstructured 1.0.1 requires langchain-core<2.0.0,>=1.0.5, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.8 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.16 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.


In [26]:
from typing import List, Optional

from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors.base import BaseDocumentCompressor
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from sentence_transformers import CrossEncoder
import torch

In [48]:

TOP_K = 10
RERANK_TOP_N = 5

# ---- Dense ----
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",   # or all-mpnet-base-v2,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Building Chroma index...")
vectorstore = Chroma.from_documents(chunks, embedding_model)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# ---- Sparse ----
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = TOP_K

# ---- Hybrid ----
hybrid_retriever = (
    EnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.3, 0.7],
    )
    | RunnableLambda(lambda docs: docs[:TOP_K])
)

# ---- Cross-encoder (simple, no class) ----
print("Loading cross-encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")

def hybrid_reranked_retrieve(query: str, top_n: int = RERANK_TOP_N):
    docs = hybrid_retriever.invoke(query)
    if not docs:
        return []
    pairs = [[query, d.page_content] for d in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    return [doc for doc, _ in ranked[:top_n]]

class FnRetriever:
    """Tiny wrapper so a plain function looks like a retriever."""
    def __init__(self, fn):
        self.fn = fn
    def invoke(self, query: str):
        return self.fn(query)

RETRIEVERS = {
    "sparse":          bm25_retriever,
    "dense":           dense_retriever,
    "hybrid":          hybrid_retriever,
    "hybrid_reranked": FnRetriever(hybrid_reranked_retrieve),
}
print("✅ Retrievers ready:", list(RETRIEVERS.keys()), "+ hybrid_reranked_retrieve()")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building Chroma index...
Loading cross-encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Retrievers ready: ['sparse', 'dense', 'hybrid', 'hybrid_reranked'] + hybrid_reranked_retrieve()


## 5. Generation Chain (Ollama)


In [42]:
# import gc
# import torch

# # Force garbage collection to free up memory from unreferenced Python objects
# gc.collect()

# # Clear PyTorch's cache if GPU or CUDA was used
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
#     gc.collect()

# print("✅ Unused system RAM and GPU VRAM cleared!")

✅ Unused system RAM and GPU VRAM cleared!


In [49]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [50]:

llm = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://127.0.0.1:11434",
    temperature=0.0,
    num_ctx=8192,
)

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context below.
If the context does not contain the answer, say:
"I could not find that information in the provided documents."

Context:
{context}

Question: {question}

Answer:"""
)
answer_chain = prompt | llm | StrOutputParser()



In [51]:
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[chunk_id={d.metadata.get('chunk_id', '?')}]\n{d.page_content}"
        for d in docs
    )

def run_rag(question: str, mode: str = "hybrid_reranked"):
    if mode == "hybrid_reranked":
        docs = hybrid_reranked_retrieve(question)
    else:
        docs = RETRIEVERS[mode].invoke(question)
    answer = answer_chain.invoke({
        "context": format_docs(docs),
        "question": question,
    })
    return {"answer": answer, "n_chunks": len(docs), "docs": docs}


### Quick smoke test


In [36]:
# Replace with a question that exists in your corpus
run_rag(
    "What was the role of the Military Revolutionary Committee under Leon Trotskii?",
    mode="hybrid_reranked",
)['answer']

'The Military Revolutionary Committee was appointed by the Soviet under Leon Trotsky to organise the socialist seizure of power.'

In [52]:
import json

# Full 50-question evaluation set (keys are integers)
ground_truth_all = {int(k): v for k, v in json.loads(r"""{
  "0": {
    "question": "What did liberals in nineteenth-century Europe mean by a 'representative, elected parliamentary government,' and why were they not considered 'democrats'?",
    "answer": "Liberals argued for a representative, elected parliamentary government, subject to laws interpreted by a well-trained judiciary that was independent of rulers and officials. They were not democrats because they did not believe in universal adult franchise; they felt men of property mainly should have the vote, and they did not want the vote for women.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      1
    ],
    "relevant_chunk_ids": [
      1,
      2
    ],
    "status": "Correct"
  },
  "1": {
    "question": "Who was Robert Owen, and what cooperative community did he attempt to build in the USA?",
    "answer": "Robert Owen (1771-1858) was a leading English manufacturer who sought to build a cooperative community called New Harmony in Indiana (USA).",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      3
    ],
    "relevant_chunk_ids": [
      5,
      6
    ],
    "status": "Correct"
  },
  "2": {
    "question": "What percentage of the Russian empire's population earned their living from agriculture at the beginning of the twentieth century, and how did this compare with France and Germany?",
    "answer": "About 85 per cent of the Russian empire's population earned their living from agriculture. In France and Germany the proportion was between 40 per cent and 50 per cent.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      5
    ],
    "relevant_chunk_ids": [
      8,
      9
    ],
    "status": "Correct"
  },
  "3": {
    "question": "What percentage of the factory labour force in Russia were women by 1914, and how did their wages compare with men's?",
    "answer": "Women made up 31 per cent of the factory labour force by 1914, but they were paid less than men (between half and three-quarters of a man's wage).",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      5
    ],
    "relevant_chunk_ids": [
      8,
      9
    ],
    "status": "Correct"
  },
  "4": {
    "question": "Under the French Constitution of 1791, what criteria distinguished an 'active citizen' entitled to vote from a 'passive citizen'?",
    "answer": "Active citizens were men above 25 years of age who paid taxes equal to at least 3 days of a labourer's wage. The remaining men and all women were classed as passive citizens.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      22
    ],
    "relevant_chunk_ids": [
      39
    ],
    "status": "Correct"
  },
  "5": {
    "question": "What was the name of the device used for executions during the Reign of Terror, and after whom was it named?",
    "answer": "The device was called the guillotine. It consisted of two poles and a blade and was named after Dr Guillotin who invented it.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      24
    ],
    "relevant_chunk_ids": [
      42,
      43
    ],
    "status": "Correct"
  },
  "6": {
    "question": "In what year did women in France finally gain the right to vote?",
    "answer": "Women in France finally won the right to vote in 1946.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      26
    ],
    "relevant_chunk_ids": [
      46,
      47
    ],
    "status": "Correct"
  },
  "7": {
    "question": "What three Caribbean colonies were important suppliers of commodities for France, and what four major commodities did they supply?",
    "answer": "The Caribbean colonies were Martinique, Guadeloupe and San Domingo. They supplied tobacco, indigo, sugar and coffee.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      27
    ],
    "relevant_chunk_ids": [
      48,
      49
    ],
    "status": "Correct"
  },
  "8": {
    "question": "What demands did the striking workers in St Petersburg make in early 1905 after the dismissals at the Putilov Iron Works?",
    "answer": "Over 110,000 workers in St Petersburg went on strike demanding a reduction of the working day to eight hours, an increase in wages, and improvement in working conditions.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      7
    ],
    "relevant_chunk_ids": [
      12,
      13
    ],
    "status": "Correct"
  },
  "9": {
    "question": "What elected body did the Tsar allow to be created during the 1905 Revolution?",
    "answer": "The Tsar allowed the creation of an elected consultative Parliament or Duma.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      8
    ],
    "relevant_chunk_ids": [
      14,
      15
    ],
    "status": "Correct"
  },
  "10": {
    "question": "How did political views on private property differ among Liberals, Radicals and Socialists in nineteenth-century Europe?",
    "answer": "Liberals supported private property and individual enterprise. Radicals were not against the existence of private property but disliked its concentration in the hands of a few. Socialists were against private property and saw it as the root of all social ills; they wanted society as a whole rather than single individuals to control property.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      1,
      3
    ],
    "relevant_chunk_ids": [
      1,
      2,
      5,
      6
    ],
    "status": "Correct"
  },
  "11": {
    "question": "How did Vladimir Lenin (Bolsheviks) and the Mensheviks differ regarding the organisational strategy of the party?",
    "answer": "Lenin (who led the Bolshevik group) thought that in a repressive society like Tsarist Russia the party should be disciplined and should control the number and quality of its members. Others (Mensheviks) thought that the party should be open to all (as in Germany).",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      7
    ],
    "relevant_chunk_ids": [
      12,
      13
    ],
    "status": "Correct"
  },
  "12": {
    "question": "How did the relationship between Russian peasants and the nobility differ from that between French peasants in Brittany and their local nobles?",
    "answer": "In Brittany during the French Revolution, peasants respected nobles and fought for them. In Russia, peasants had no respect for the nobility because nobles got their power and position through their services to the Tsar, not through local popularity. Russian peasants frequently refused to pay rent and even murdered landlords (on a large scale in 1902 and all over Russia in 1905).",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      6
    ],
    "relevant_chunk_ids": [
      10,
      11
    ],
    "status": "Correct"
  },
  "13": {
    "question": "What were the main problems faced by industrial workers in Russia at the beginning of the twentieth century?",
    "answer": "Work hours were often long (sometimes 15 hours in craft units compared with 10 or 12 in factories), wages were poor, unemployment was common, housing and sanitation were problems, and women were paid only between half and three-quarters of a man's wage. Factory inspectors could not prevent rules being broken.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      5
    ],
    "relevant_chunk_ids": [
      8,
      9
    ],
    "status": "Correct"
  },
  "14": {
    "question": "What sequence of events in Petrograd in February 1917 led to the formation of the Petrograd Soviet?",
    "answer": "On 22 February a lockout took place at a factory. The next day workers in fifty factories called a strike in sympathy (International Women's Day). Demonstrations grew; the government imposed a curfew and suspended the Duma on 25 February. On 27 February the Police Headquarters were ransacked, regiments mutinied and joined the striking workers, and the Petrograd Soviet was formed that evening.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      9,
      10
    ],
    "relevant_chunk_ids": [
      16,
      17,
      18,
      19
    ],
    "status": "Correct"
  },
  "15": {
    "question": "What were the main features of the Constitution of 1791 in France?",
    "answer": "The Constitution of 1791 limited the powers of the monarch and separated them among the legislature, executive and judiciary, making France a constitutional monarchy. The power to make laws was vested in the National Assembly, which was indirectly elected. Only active citizens (men above 25 who paid taxes equal to at least 3 days of a labourer's wage) could vote; the remaining men and all women were passive citizens.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      22
    ],
    "relevant_chunk_ids": [
      39
    ],
    "status": "Correct"
  },
  "16": {
    "question": "How did Tsar Nicholas II limit the power of the Duma after the 1905 Revolution?",
    "answer": "The Tsar dismissed the first Duma within 75 days and the re-elected second Duma within three months. He changed the voting laws and packed the third Duma with conservative politicians; liberals and revolutionaries were kept out.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      8
    ],
    "relevant_chunk_ids": [
      14,
      15
    ],
    "status": "Correct"
  },
  "17": {
    "question": "Who were the 'sans-culottes', what did their name literally mean, and how did their attire symbolise their political identity?",
    "answer": "The sans-culottes were Jacobin members from the less prosperous sections of society (small shopkeepers, artisans, servants and daily-wage workers). The name literally means 'those without knee breeches'. They wore long striped trousers similar to those worn by dock workers and red caps that symbolised liberty, to set themselves apart from the nobles who wore knee breeches.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      23,
      24
    ],
    "relevant_chunk_ids": [
      40,
      41,
      42,
      43
    ],
    "status": "Correct"
  },
  "18": {
    "question": "What happened on Bloody Sunday in 1905 and what were its immediate consequences?",
    "answer": "A procession of workers led by Father Gapon reached the Winter Palace and was attacked by the police and the Cossacks. Over 100 workers were killed and about 300 wounded. The incident started a series of events known as the 1905 Revolution: strikes took place all over the country, universities closed down, and middle-class professionals established the Union of Unions and demanded a constituent assembly.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      7
    ],
    "relevant_chunk_ids": [
      12,
      13
    ],
    "status": "Correct"
  },
  "19": {
    "question": "What mandatory measures did Robespierre enforce regarding food, forms of address and churches during the Reign of Terror?",
    "answer": "Meat and bread were rationed; the use of white flour was forbidden and all citizens were required to eat pain d'égalité (equality bread made of wholewheat). Instead of Monsieur and Madame, all French men and women were to be addressed as Citoyen and Citoyenne. Churches were shut down and their buildings converted into barracks or offices.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      24
    ],
    "relevant_chunk_ids": [
      42,
      43
    ],
    "status": "Correct"
  },
  "20": {
    "question": "How did Russian peasants traditionally manage land, and what demand did the Socialist Revolutionary Party make?",
    "answer": "Russian peasants pooled their land together periodically and their commune (mir) divided it according to the needs of individual families. The Socialist Revolutionary Party struggled for peasants' rights and demanded that land belonging to nobles be transferred to peasants.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      6
    ],
    "relevant_chunk_ids": [
      10,
      11
    ],
    "status": "Correct"
  },
  "21": {
    "question": "What economic controls did Robespierre's government impose during the Reign of Terror (1793–1794)?",
    "answer": "The government issued laws placing a maximum ceiling on wages and prices. Meat and bread were rationed. Peasants were forced to transport their grain to the cities and sell it at prices fixed by the government. The use of more expensive white flour was forbidden; all citizens had to eat pain d'égalité.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      24
    ],
    "relevant_chunk_ids": [
      42,
      43
    ],
    "status": "Correct"
  },
  "22": {
    "question": "How did foreign powers react to the French Revolution in 1792, and what was the French response?",
    "answer": "Rulers of other neighbouring countries were worried by the developments in France and made plans to send troops. Before this could happen, the National Assembly voted in April 1792 to declare war against Prussia and Austria.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      23
    ],
    "relevant_chunk_ids": [
      40,
      41
    ],
    "status": "Correct"
  },
  "23": {
    "question": "Trace the legal status of slavery in the French colonies from the eighteenth century to 1848.",
    "answer": "Throughout the eighteenth century there was little criticism of slavery in France. The Convention in 1794 legislated to free all slaves in the French overseas possessions. Ten years later Napoleon reintroduced slavery. Slavery was finally abolished in French colonies in 1848.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      27
    ],
    "relevant_chunk_ids": [
      48,
      49
    ],
    "status": "Correct"
  },
  "24": {
    "question": "What political and social gains did women make during the early years of the French Revolution, and when did they finally obtain the vote?",
    "answer": "Women formed political clubs (about sixty clubs came up; the Society of Revolutionary and Republican Women was the most famous). Early revolutionary laws made schooling compulsory for girls, made marriage a contract entered into freely, legalised divorce for both women and men, and allowed women to train for jobs. They were classed as passive citizens in 1791 and their clubs were banned during the Reign of Terror. Women in France won the right to vote in 1946.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      25,
      26
    ],
    "relevant_chunk_ids": [
      44,
      45,
      46,
      47
    ],
    "status": "Correct"
  },
  "25": {
    "question": "How did freedom of the press change in France after the storming of the Bastille?",
    "answer": "One important law that came into effect soon after the storming of the Bastille was the abolition of censorship. The Declaration of the Rights of Man and Citizen proclaimed freedom of speech and expression to be a natural right. Newspapers, pamphlets, books and printed pictures flooded the towns of France and travelled rapidly into the countryside.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      28
    ],
    "relevant_chunk_ids": [
      50,
      51
    ],
    "status": "Correct"
  },
  "26": {
    "question": "How did the voting franchise change in France from the Estates General of 1789 to the National Convention of 1792?",
    "answer": "In the Estates General voting was by estate. The Constitution of 1791 created an indirectly elected National Assembly in which only active citizens (men above 25 who paid taxes equal to at least 3 days of a labourer's wage) could vote. In September 1792 the newly elected Convention abolished the monarchy and granted the right to vote to all men of 21 years and above, regardless of wealth.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      21,
      22,
      24
    ],
    "relevant_chunk_ids": [
      37,
      38,
      39,
      42,
      43
    ],
    "status": "Correct"
  },
  "27": {
    "question": "What were the human and economic costs of Russia's involvement in the First World War up to 1917?",
    "answer": "There were over 7 million casualties by 1917. As the Russian army retreated it destroyed crops and buildings, leading to over 3 million refugees. Railway lines began to break down, able-bodied men were called up creating labour shortages, and by the winter of 1916 bread and flour became scarce in the cities, with riots at bread shops becoming common.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      8,
      9
    ],
    "relevant_chunk_ids": [
      14,
      15,
      16,
      17
    ],
    "status": "Correct"
  },
  "28": {
    "question": "What international body did socialists form in the 1870s to coordinate their efforts across Europe?",
    "answer": "By the 1870s socialist ideas had spread through Europe and socialists formed an international body — namely, the Second International.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      4
    ],
    "relevant_chunk_ids": [
      7
    ],
    "status": "Correct"
  },
  "29": {
    "question": "How did Indian thinkers respond to the ideas of the French Revolution according to the text?",
    "answer": "In India, Raja Rammohan Roy and Derozio talked of the significance of the French Revolution. Tipu Sultan and Rammohan Roy are given as examples of individuals who responded to the ideas coming from revolutionary France. Nationalists elsewhere, including India, also read the writings of Giuseppe Mazzini.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      0,
      2,
      28
    ],
    "relevant_chunk_ids": [
      0,
      3,
      4,
      50,
      51
    ],
    "status": "Correct"
  },
  "30": {
    "question": "Who was Giuseppe Mazzini and what role did Italian nationalists play after 1815?",
    "answer": "Giuseppe Mazzini was an Italian nationalist who, after 1815, conspired with others to achieve a nation where all citizens would have equal rights, in Italy. Nationalists elsewhere — including India — read his writings.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      2
    ],
    "relevant_chunk_ids": [
      3,
      4
    ],
    "status": "Correct"
  },
  "31": {
    "question": "What specific events occurred during 'Bloody Sunday' in 1905, who led the procession, and what were the immediate casualties?",
    "answer": "A procession of workers led by Father Gapon reached the Winter Palace and was attacked by the police and the Cossacks. Over 100 workers were killed and about 300 wounded. This incident started a series of events that became known as the 1905 Revolution.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      7
    ],
    "relevant_chunk_ids": [
      12,
      13
    ],
    "status": "Correct"
  },
  "32": {
    "question": "What was the Directory in post-Jacobin France and why was it structured with five members?",
    "answer": "After the fall of the Jacobin government a new constitution provided for two elected legislative councils that appointed a Directory, an executive made up of five members. This was meant as a safeguard against the concentration of power in a one-man executive as under the Jacobins. The Directors often clashed with the legislative councils, and the political instability paved the way for the rise of Napoleon Bonaparte.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      25
    ],
    "relevant_chunk_ids": [
      44,
      45
    ],
    "status": "Correct"
  },
  "33": {
    "question": "What panic swept through the French countryside in the summer of 1789, and how did peasants react?",
    "answer": "Rumours spread from village to village that the lords of the manor had hired bands of brigands who were on their way to destroy the ripe crops. Caught in a frenzy of fear, peasants in several districts seized hoes and pitchforks and attacked châteaux. They looted hoarded grain and burnt down documents containing records of manorial dues.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      21
    ],
    "relevant_chunk_ids": [
      37,
      38
    ],
    "status": "Correct"
  },
  "34": {
    "question": "Who was Abbé Sieyès and what influential pamphlet did he write?",
    "answer": "Abbé Sieyès, originally a priest, wrote an influential pamphlet called What is the Third Estate?",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      21
    ],
    "relevant_chunk_ids": [
      37,
      38
    ],
    "status": "Correct"
  },
  "35": {
    "question": "What was the 'Marseillaise', who composed it, and how did it become associated with the French Revolution?",
    "answer": "The Marseillaise was a patriotic song composed by the poet Roget de L'Isle. It was sung for the first time by volunteers from Marseilles as they marched into Paris and so got its name. It is now the national anthem of France.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      23
    ],
    "relevant_chunk_ids": [
      40,
      41
    ],
    "status": "Correct"
  },
  "36": {
    "question": "What happened to Louis XVI after France became a republic?",
    "answer": "Louis XVI was sentenced to death by a court on the charge of treason. On 21 January 1793 he was executed publicly at the Place de la Concorde. The queen Marie Antoinette met with the same fate shortly after.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      24
    ],
    "relevant_chunk_ids": [
      42,
      43
    ],
    "status": "Correct"
  },
  "37": {
    "question": "What political clubs did women form during the French Revolution, what were their main demands, and how did the government respond during the Reign of Terror?",
    "answer": "Women started their own political clubs and newspapers; about sixty women's clubs came up in different French cities. The Society of Revolutionary and Republican Women was the most famous. Their main demand was that women enjoy the same political rights as men (the right to vote, to be elected to the Assembly and to hold political office). During the Reign of Terror the government ordered the closure of women's clubs and banned their political activities; many prominent women were arrested and a number executed.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      25,
      26
    ],
    "relevant_chunk_ids": [
      44,
      45,
      46,
      47
    ],
    "status": "Correct"
  },
  "38": {
    "question": "How did Russian peasants differ from other European peasants in the way they managed land?",
    "answer": "Russian peasants were different from other European peasants in that they pooled their land together periodically and their commune (mir) divided it according to the needs of individual families.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      6
    ],
    "relevant_chunk_ids": [
      10,
      11
    ],
    "status": "Correct"
  },
  "39": {
    "question": "What was the impact of the First World War on Russian industry and urban food supply by the winter of 1916?",
    "answer": "Russia's own industries were few and the country was cut off from other suppliers by German control of the Baltic Sea. Industrial equipment disintegrated, railway lines began to break down, able-bodied men were called up creating labour shortages, and large supplies of grain were sent to feed the army. By the winter of 1916 bread and flour became scarce in the cities and riots at bread shops were common.",
    "document": "TEST_HISTORY_RAG_PDF.pdf",
    "relevant_pages": [
      8,
      9
    ],
    "relevant_chunk_ids": [
      14,
      15,
      16,
      17
    ],
    "status": "Correct"
  },
  "40": {
    "question": "What is the core rule regarding the fundamental purpose of data tables according to accessibility guidelines?",
    "answer": "Use tables to organize data, not to format information. Never use a table for layout. Simple tables are easier to interpret for all students and screen readers.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      0
    ],
    "relevant_chunk_ids": [
      52
    ],
    "status": "Correct"
  },
  "41": {
    "question": "What four structural cell formatting practices must be avoided when designing accessible tables?",
    "answer": "Avoid merging cells, split cells, nested cells within a cell, and blank cells.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      0
    ],
    "relevant_chunk_ids": [
      52
    ],
    "status": "Correct"
  },
  "42": {
    "question": "Instead of using a table to format course information layout, what alternative formatting methods are recommended?",
    "answer": "Use tabs position or columns format.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      0
    ],
    "relevant_chunk_ids": [
      52
    ],
    "status": "Correct"
  },
  "43": {
    "question": "When a table heading like 'Grading Scale' is placed inside a merged cell, what is the best-practice solution?",
    "answer": "Move the heading 'Grading Scale' out of the table instead of merging cells.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      0
    ],
    "relevant_chunk_ids": [
      52
    ],
    "status": "Correct"
  },
  "44": {
    "question": "Why should hyphens or dashes be avoided in grade ranges, and how should they be formatted for screen readers?",
    "answer": "Screen readers do not know whether the '-' symbol is a minus, hyphen, dash, en dash or em dash, so it will not be read. Ranges should be written with words such as 'to' or 'minus' (e.g., '960 to 1000', 'A minus').",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      1
    ],
    "relevant_chunk_ids": [
      53
    ],
    "status": "Correct"
  },
  "45": {
    "question": "How should a single table containing mixed categories (Reading Assignments and Writing Projects) be restructured for accessibility?",
    "answer": "Separate them into two distinct tables, each with its own header row.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      1
    ],
    "relevant_chunk_ids": [
      53
    ],
    "status": "Correct"
  },
  "46": {
    "question": "What is the primary accessibility hazard of using colour alone to indicate project status in a data table?",
    "answer": "Using colour alone fails to convey information to colour-blind users (for whom completed and incomplete items may appear the same) and to screen-reader users who cannot perceive colour.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      2
    ],
    "relevant_chunk_ids": [
      54
    ],
    "status": "Correct"
  },
  "47": {
    "question": "In a project-completion tracking table, what alternative textual cues should be used instead of colour indicators?",
    "answer": "Provide explicit textual cues such as 'Y or Yes' for completed items and 'N or No' for incomplete items.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      2
    ],
    "relevant_chunk_ids": [
      54
    ],
    "status": "Correct"
  },
  "48": {
    "question": "In the sample 'Grading Scale' table, what point and percentage boundaries define an 'A plus' grade versus an 'A' grade?",
    "answer": "An 'A plus' grade is 960 to 1000 points (96 to 100 %). An 'A' grade is 930 to 959 points (93 to 95 %).",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      0,
      1
    ],
    "relevant_chunk_ids": [
      52,
      53
    ],
    "status": "Correct"
  },
  "49": {
    "question": "In Example 3 (Reading Assignments and Writing Projects), what are the exact due dates for Reading 1, Reading 2, Writing 1 and Writing 2?",
    "answer": "Reading 1: 7/22/2021; Reading 2: 7/23/2021; Writing 1: 8/1/2021; Writing 2: 9/1/2021.",
    "document": "Table-Examples.pdf",
    "relevant_pages": [
      1
    ],
    "relevant_chunk_ids": [
      53
    ],
    "status": "Correct"
  }
}""").items()}

print(f"Evaluation set size: {len(ground_truth_all)} questions")
print("Sample q0:", ground_truth_all[0]["question"][:80], "...")

Evaluation set size: 50 questions
Sample q0: What did liberals in nineteenth-century Europe mean by a 'representative, electe ...


## 7. Batch Retrieval + Generation


In [53]:
import pandas as pd
from tqdm.auto import tqdm

records = []
for qid, gt in tqdm(ground_truth_all.items(), desc="Questions"):
    question = gt["question"]
    reference = gt["answer"]
    relevant_ids = gt.get("relevant_chunk_ids", [])

    reference_contexts = []
    for cid in relevant_ids:
        if 0 <= cid < len(chunks):
            reference_contexts.append(chunks[cid].page_content)

    for mode, retriever in RETRIEVERS.items():
        docs = retriever.invoke(question)
        context_str = format_docs(docs)
        try:
            response = answer_chain.invoke(
                {"context": context_str, "question": question}
            )
        except Exception as e:
            response = f"[GENERATION_ERROR: {type(e).__name__}: {e}]"

        records.append({
            "qid": qid,
            "question": question,
            "mode": mode,
            "ground_truth": reference,
            "prediction": response,
            "reference_contexts": reference_contexts,
            "relevant_chunk_ids": relevant_ids,
            "retrieved_contexts": [d.page_content for d in docs],
            "retrieved_sources": [
                f"{d.metadata.get('source', '?')}#p{d.metadata.get('page', '?')}"
                for d in docs
            ],
            "n_chunks": len(docs),
        })

df_eval = pd.DataFrame(records)
print(f"✅ {len(df_eval)} rows "
      f"({len(ground_truth_all)} questions × {len(RETRIEVERS)} modes)")
print("\nChunks per mode:")
print(df_eval.groupby("mode")["n_chunks"].agg(["min", "max", "mean"]).round(2))
errs = df_eval["prediction"].str.startswith("[GENERATION_ERROR")
print(f"\nGeneration errors: {errs.sum()}/{len(df_eval)}")

Questions:   0%|          | 0/50 [00:00<?, ?it/s]

✅ 200 rows (50 questions × 4 modes)

Chunks per mode:
                 min  max  mean
mode                           
dense             10   10  10.0
hybrid            10   10  10.0
hybrid_reranked    5    5   5.0
sparse            10   10  10.0

Generation errors: 0/200


## 8. Lightweight Metrics (no external API required)


In [54]:
import re
import numpy as np
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def tokenize(t: str):
    return re.findall(r"\b\w+\b", str(t).lower())

def token_f1(pred, ref):
    p, r = Counter(tokenize(pred)), Counter(tokenize(ref))
    if not p or not r:
        return 0.0
    ov = sum((p & r).values())
    if ov == 0:
        return 0.0
    prec, rec = ov / sum(p.values()), ov / sum(r.values())
    return 2 * prec * rec / (prec + rec)

def exact_match(pred, ref):
    return float(str(pred).strip().lower() == str(ref).strip().lower())

_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_sim(a, b):
    v = _embedder.encode([str(a), str(b)], normalize_embeddings=True)
    return float(cosine_similarity([v[0]], [v[1]])[0][0])

try:
    from rouge_score import rouge_scorer
    _rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    HAVE_ROUGE = True
except ImportError:
    HAVE_ROUGE = False

try:
    import nltk
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    try:
        nltk.data.find("tokenizers/punkt")
    except LookupError:
        nltk.download("punkt", quiet=True)
        nltk.download("punkt_tab", quiet=True)
    _smooth = SmoothingFunction().method1
    HAVE_BLEU = True
except ImportError:
    HAVE_BLEU = False

def score_row(pred, ref):
    out = {
        "exact_match": exact_match(pred, ref),
        "token_f1": token_f1(pred, ref),
        "semantic_sim": semantic_sim(pred, ref),
    }
    if HAVE_ROUGE:
        s = _rouge.score(ref, pred)
        out.update({
            "rouge1": s["rouge1"].fmeasure,
            "rouge2": s["rouge2"].fmeasure,
            "rougeL": s["rougeL"].fmeasure,
        })
    else:
        out.update({"rouge1": np.nan, "rouge2": np.nan, "rougeL": np.nan})
    out["bleu"] = (
        sentence_bleu([tokenize(ref)], tokenize(pred), smoothing_function=_smooth)
        if HAVE_BLEU else np.nan
    )
    return out

print("Metric helpers ready (ROUGE=", HAVE_ROUGE, ", BLEU=", HAVE_BLEU, ")")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Metric helpers ready (ROUGE= True , BLEU= True )


In [55]:
from tqdm.auto import tqdm
tqdm.pandas()

scored = df_eval.progress_apply(
    lambda r: pd.Series(score_row(r["prediction"], r["ground_truth"])),
    axis=1,
)
df_scored = pd.concat([df_eval, scored], axis=1)

METRICS = ["exact_match", "token_f1", "semantic_sim", "rouge1", "rouge2", "rougeL", "bleu"]
summary = (
    df_scored.groupby("mode")[METRICS]
    .agg(["mean", "std", "count"])
    .round(4)
)
print(summary)

  0%|          | 0/200 [00:00<?, ?it/s]

                exact_match            token_f1               semantic_sim  \
                       mean  std count     mean     std count         mean   
mode                                                                         
dense                   0.0  0.0    50   0.5704  0.2238    50       0.7881   
hybrid                  0.0  0.0    50   0.5704  0.2174    50       0.7876   
hybrid_reranked         0.0  0.0    50   0.5588  0.2323    50       0.7769   
sparse                  0.0  0.0    50   0.5245  0.2819    50       0.7289   

                               rouge1  ...        rouge2                \
                    std count    mean  ... count    mean     std count   
mode                                   ...                               
dense            0.1958    50  0.5835  ...    50  0.4283  0.2578    50   
hybrid           0.1926    50  0.5860  ...    50  0.4302  0.2455    50   
hybrid_reranked  0.2134    50  0.5747  ...    50  0.4203  0.2594    50   
sparse   

## 9. Optional LLM-as-Judge (local Ollama)

Uses the same `qwen2.5:3b` model. Returns correctness & faithfulness on a 1–5 scale.


In [56]:
import json
import re

JUDGE_SYSTEM = (
    "You are a strict evaluator of RAG answers. "
    "Compare the PREDICTION to the GROUND TRUTH. "
    "Scoring rubric (integers only):\n"
    "  correctness: 5=fully correct, 4=mostly correct minor omissions, "
    "3=partially correct, 2=mostly wrong, 1=wrong/unrelated.\n"
    "  faithfulness: 5=entirely supported by CONTEXT, "
    "4=supported with minor extrapolation, 3=partial support, "
    "2=mostly unsupported, 1=contradicts context.\n"
    "Return ONLY compact JSON: "
    '{{"correctness": <int 1-5>, "faithfulness": <int 1-5>, "reason": "<short>"}}'
)

JUDGE_HUMAN = """QUESTION:
{q}

GROUND TRUTH:
{gt}

CONTEXT (retrieved):
{ctx}

PREDICTION:
{pred}
"""

judge_prompt = ChatPromptTemplate.from_messages(
    [("system", JUDGE_SYSTEM), ("human", JUDGE_HUMAN)]
)
judge_chain = judge_prompt | llm | StrOutputParser()

_JSON_RE = re.compile(r"\{.*\}", re.S)

def _fmt_ctx(row, limit_chars=4000):
    ctxs = row.get("retrieved_contexts", [])
    if not isinstance(ctxs, list):
        return ""
    text = "\n---\n".join(str(c) for c in ctxs)
    return text[:limit_chars]

def judge_row(row):
    try:
        raw = judge_chain.invoke({
            "q": row["question"],
            "gt": row["ground_truth"],
            "ctx": _fmt_ctx(row),
            "pred": row["prediction"],
        })
        m = _JSON_RE.search(raw or "")
        if not m:
            return dict(
                llm_correctness=np.nan,
                llm_faithfulness=np.nan,
                llm_reason=(raw or "")[:160],
            )
        data = json.loads(m.group(0))
        return dict(
            llm_correctness=float(data.get("correctness", np.nan)),
            llm_faithfulness=float(data.get("faithfulness", np.nan)),
            llm_reason=str(data.get("reason", ""))[:240],
        )
    except Exception as e:
        return dict(
            llm_correctness=np.nan,
            llm_faithfulness=np.nan,
            llm_reason=f"err: {type(e).__name__}: {e}"[:240],
        )

print("Judge chain ready. Prompt vars:", judge_prompt.input_variables)

Judge chain ready. Prompt vars: ['ctx', 'gt', 'pred', 'q']


In [57]:
# Run judge (one LLM call per row – can be slow on CPU)
judged = df_scored.progress_apply(lambda r: pd.Series(judge_row(r)), axis=1)
df_scored = pd.concat([df_scored, judged], axis=1)

print(df_scored[["qid", "mode", "token_f1", "semantic_sim",
                 "llm_correctness", "llm_faithfulness"]].head(12))

  0%|          | 0/200 [00:00<?, ?it/s]

    qid             mode  token_f1  semantic_sim  llm_correctness  \
0     0           sparse  0.658683      0.844882              5.0   
1     0            dense  0.506329      0.837267              5.0   
2     0           hybrid  0.675676      0.853098              5.0   
3     0  hybrid_reranked  0.546584      0.822313              5.0   
4     1           sparse  0.952381      0.975407              5.0   
5     1            dense  1.000000      0.999270              5.0   
6     1           hybrid  1.000000      0.999270              5.0   
7     1  hybrid_reranked  1.000000      0.999270              5.0   
8     2           sparse  0.586667      0.946100              5.0   
9     2            dense  0.647887      0.950222              5.0   
10    2           hybrid  0.647887      0.950222              5.0   
11    2  hybrid_reranked  0.657534      0.955364              5.0   

    llm_faithfulness  
0                4.0  
1                5.0  
2                5.0  
3         

In [58]:
FULL_METRICS = METRICS + ["llm_correctness", "llm_faithfulness"]
full_summary = (
    df_scored.groupby("mode")[FULL_METRICS]
    .agg(["mean", "std", "count"])
    .round(4)
)
print(full_summary)

                exact_match            token_f1               semantic_sim  \
                       mean  std count     mean     std count         mean   
mode                                                                         
dense                   0.0  0.0    50   0.5704  0.2238    50       0.7881   
hybrid                  0.0  0.0    50   0.5704  0.2174    50       0.7876   
hybrid_reranked         0.0  0.0    50   0.5588  0.2323    50       0.7769   
sparse                  0.0  0.0    50   0.5245  0.2819    50       0.7289   

                               rouge1  ... rougeL    bleu                \
                    std count    mean  ...  count    mean     std count   
mode                                   ...                                
dense            0.1958    50  0.5835  ...     50  0.3225  0.2491    50   
hybrid           0.1926    50  0.5860  ...     50  0.3207  0.2325    50   
hybrid_reranked  0.2134    50  0.5747  ...     50  0.3134  0.2499    50   
spa

## 10. Save Results


In [59]:
import json
from pathlib import Path

OUT_DIR = Path("/content/rag_exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)

pkl_path = OUT_DIR / "df_scored.pkl"
df_scored.to_pickle(pkl_path)
print(f"✅ Pickle → {pkl_path}")

csv_path = OUT_DIR / "df_scored.csv"
df_csv = df_scored.copy()
for col in ["retrieved_contexts", "retrieved_sources",
            "reference_contexts", "relevant_chunk_ids"]:
    if col in df_csv.columns:
        df_csv[col] = df_csv[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, (list, tuple)) else x
        )
df_csv.to_csv(csv_path, index=False)
print(f"✅ CSV    → {csv_path}")

try:
    from google.colab import files
    files.download(str(pkl_path))
    files.download(str(csv_path))
except ImportError:
    pass

✅ Pickle → /content/rag_exports/df_scored.pkl
✅ CSV    → /content/rag_exports/df_scored.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>